In [1]:
import json
import re
from collections import Counter
import numpy as np
import pandas as pd

In [2]:
# Đọc 3 file .jsonl vừa tạo ở Bước 3
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_data = read_jsonl("data/processed/train.jsonl")
val_data   = read_jsonl("data/processed/val.jsonl")
test_data  = read_jsonl("data/processed/test.jsonl")

In [3]:
# =====================================================================
# HÀM EDA 1: Thống kê số lượng câu, % không có đáp án và độ dài (từ)
# =====================================================================
def eda_summary_table(splits_dict: dict) -> pd.DataFrame:
    """
    Đầu vào: Dictionary chứa tên tập dữ liệu và danh sách các mẫu của tập đó.
    Đầu ra: Bảng DataFrame tổng hợp mọi con số cần đưa vào Proposal ngày 7/11.
    """
    rows = []
    for split_name, samples in splits_dict.items():
        n_total = len(samples)
        n_articles = len({s["title"] for s in samples})
        n_contexts = len({s["context"] for s in samples})

        # Đếm số câu không có đáp án (is_impossible = True) và có đáp án
        n_noans = sum(1 for s in samples if s["is_impossible"])
        n_hasans = n_total - n_noans

        # Tính độ dài (đếm số âm tiết/từ cách nhau bởi khoảng trắng)
        ctx_lens = [len(s["context"].split()) for s in samples]
        q_lens   = [len(s["question"].split()) for s in samples]
        
        # Độ dài đáp án chỉ tính trên các câu CÓ đáp án (HasAns)
        ans_lens = [
            len(s["answers"]["text"][0].split())
            for s in samples if (not s["is_impossible"]) and len(s["answers"]["text"]) > 0
        ]

        rows.append({
            "Tập": split_name,
            "Số bài viết": n_articles,
            "Số đoạn văn": n_contexts,
            "Tổng câu hỏi": n_total,
            "Có đáp án (HasAns)": n_hasans,
            "Không đáp án (NoAns)": n_noans,
            "% NoAns": round(n_noans * 100.0 / n_total, 2),
            "Dài Context TB (từ)": round(float(np.mean(ctx_lens)), 1),
            "Dài Context P95 (từ)": int(np.percentile(ctx_lens, 95)),
            "Dài Câu hỏi TB (từ)": round(float(np.mean(q_lens)), 1),
            "Dài Đáp án TB (từ)": round(float(np.mean(ans_lens)), 1) if ans_lens else 0,
            "Dài Đáp án P95 (từ)": int(np.percentile(ans_lens, 95)) if ans_lens else 0,
        })

    return pd.DataFrame(rows)


df_overview = eda_summary_table({
    "Train (90%)": train_data,
    "Val (10%)": val_data,
    "Test (Official Dev)": test_data
})
print("=== BẢNG 1: THỐNG KÊ TỔNG QUAN (ĐƯA VÀO PROPOSAL) ===")
print(df_overview.to_string(index=False))

=== BẢNG 1: THỐNG KÊ TỔNG QUAN (ĐƯA VÀO PROPOSAL) ===
                Tập  Số bài viết  Số đoạn văn  Tổng câu hỏi  Có đáp án (HasAns)  Không đáp án (NoAns)  % NoAns  Dài Context TB (từ)  Dài Context P95 (từ)  Dài Câu hỏi TB (từ)  Dài Đáp án TB (từ)  Dài Đáp án P95 (từ)
        Train (90%)          124         3774         20930               14128                  6802    32.50                181.1                   315                 14.7                 9.9                   31
          Val (10%)           14          327          1835                1260                   575    31.34                179.6                   324                 14.5                11.1                   33
Test (Official Dev)          138         3171          5692                3852                  1840    32.33                179.1                   311                 14.6                 9.8                   31


In [4]:
# =====================================================================
# HÀM EDA 2: Phân loại câu hỏi tiếng Việt (Ai, Cái gì, Khi nào, Vì sao...)
# =====================================================================
def eda_question_types(samples: list) -> pd.DataFrame:
    """
    Phân loại câu hỏi dựa vào từ để hỏi tiếng Việt.
    Hàm này dùng cho cả EDA tuần 1 lẫn Phân tích lỗi (Thí nghiệm E3) ở tuần 3.
    """
    patterns = [
        ("Khi nào (Thời gian)", r"\b(khi nào|năm nào|tháng nào|ngày nào|thời gian nào|thế kỷ nào|bao lâu|lúc nào)\b"),
        ("Ở đâu (Địa điểm)", r"\b(ở đâu|nơi nào|tại đâu|vùng nào|quốc gia nào|nước nào|tỉnh nào|thành phố nào)\b"),
        ("Ai (Con người)", r"\b(ai là|ai đã|những ai|của ai|với ai|do ai|\bai\b)\b"),
        ("Bao nhiêu (Số lượng)", r"\b(bao nhiêu|mấy|tỉ lệ|số lượng|diện tích|chiều dài|độ cao)\b"),
        ("Vì sao (Nguyên nhân)", r"\b(vì sao|tại sao|do đâu|lý do gì|nguyên nhân nào)\b"),
        ("Như thế nào (Cách thức)", r"\b(như thế nào|bằng cách nào|ra sao|thế nào)\b"),
        ("Cái gì (Sự vật/Khái niệm)", r"\b(là gì|cái gì|điều gì|việc gì|loại nào|ngành nào|tên là gì|gọi là gì)\b"),
    ]

    def get_qtype(question_text: str) -> str:
        q_low = question_text.lower()
        for label, regex in patterns:
            if re.search(regex, q_low):
                return label
        return "Khác"

    counts = Counter(get_qtype(s["question"]) for s in samples)
    total = len(samples)

    table = [
        {
            "Loại câu hỏi": qtype,
            "Số lượng": cnt,
            "Tỉ lệ (%)": round(cnt * 100.0 / total, 2)
        }
        for qtype, cnt in counts.most_common()
    ]
    return pd.DataFrame(table)


df_qtypes = eda_question_types(train_data)
print("\n=== BẢNG 2: PHÂN BỐ LOẠI CÂU HỎI TRÊN TẬP TRAIN ===")
print(df_qtypes.to_string(index=False))


=== BẢNG 2: PHÂN BỐ LOẠI CÂU HỎI TRÊN TẬP TRAIN ===
             Loại câu hỏi  Số lượng  Tỉ lệ (%)
                     Khác      8133      38.86
Cái gì (Sự vật/Khái niệm)      2938      14.04
     Bao nhiêu (Số lượng)      2179      10.41
           Ai (Con người)      1797       8.59
      Khi nào (Thời gian)      1737       8.30
  Như thế nào (Cách thức)      1730       8.27
         Ở đâu (Địa điểm)      1345       6.43
     Vì sao (Nguyên nhân)      1071       5.12
